# =========================================
# STEP 1. Install libraries + mount Google Drive
# =========================================

In [1]:
# =========================================
# STEP 1. Install libraries + mount Google Drive
# =========================================
!pip -q install ultralytics scikit-learn

from google.colab import drive
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.6 MB/s eta 0:00:00
Mounted at /content/drive


# =========================================
# STEP 2. FreshGuard EfficientNet-B0 multitask training
#  - Use the Food Freshness Dataset stored in Drive
#  - 10 fruit types + 3 freshness levels (fresh/normal/rotten)
#  - Resulting weights are saved to Drive
# =========================================

# Currently, running STEP 2 and STEP 3 produces the results

In [ ]:
# ============================================
# STEP 0. Install + mount Google Drive
# ============================================
!pip -q install ultralytics scikit-learn

from google.colab import drive
drive.mount('/content/drive')

# ============================================
# Common imports / path settings
# ============================================
import os
import zipfile
import shutil
import uuid
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRINT_EVERY = 50
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ===== Path of the Kaggle zip inside Drive =====
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/datasets/food-freshness-dataset.zip")

# ===== Local path to extract the zip into =====
RAW_ROOT    = Path("/content/data/raw")
EXTRACT_DIR = RAW_ROOT / "food_freshness"

# ===== EfficientNet weight save location (Drive) =====
EFFNET_OUT_DIR  = Path("/content/drive/MyDrive/FreshGuard/effnet")
EFFNET_OUT_DIR.mkdir(parents=True, exist_ok=True)
EFFNET_BEST_PATH = EFFNET_OUT_DIR / "efficientnet_b0_freshguard_multitask.pt"

# ===== Preprocessing paths inside Colab =====
WORK_DATA_DIR = Path("/content/data/freshguard")
IMG_ROOT      = WORK_DATA_DIR / "images"
META_CSV      = WORK_DATA_DIR / "labels.csv"
WORK_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Fruits to use (10 types)
TARGET_FRUITS = {
    "apple":       ["apple"],
    "banana":      ["banana"],
    "bell_pepper": ["bellpepper", "bell pepper", "bell_pepper", "capsicum"],
    "carrot":      ["carrot"],
    "cucumber":    ["cucumber"],
    "mango":       ["mango"],
    "orange":      ["orange"],
    "potato":      ["potato"],
    "strawberry":  ["strawberry"],
    "tomato":      ["tomato", "tomatos"],
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# ============================================
# STEP 1. Extract the zip from Drive → /content
# ============================================
def resolve_source_dir() -> Path:
    """
    1) If /content/data/raw/food_freshness already exists, use it
    2) Otherwise, extract the Drive ZIP into /content/data/raw/food_freshness and use it
    """
    if EXTRACT_DIR.exists():
        print(f"[Data] Using local folder: {EXTRACT_DIR}")
        return EXTRACT_DIR

    if not DRIVE_ZIP_PATH.exists():
        raise RuntimeError(f"Zip not found in Drive. Check the path first: {DRIVE_ZIP_PATH}")

    print(f"[Data] Using Drive zip: {DRIVE_ZIP_PATH}")
    RAW_ROOT.mkdir(parents=True, exist_ok=True)

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    print(f"[Data] Zip extraction complete: {EXTRACT_DIR}")
    return EXTRACT_DIR

# ============================================
# STEP 2. Label inference (fruit / Fresh vs Rotten)
# ============================================
def infer_fruit(path: Path):
    joined = " ".join(p.lower() for p in path.parts)
    for canon, keywords in TARGET_FRUITS.items():
        for k in keywords:
            if k in joined:
                return canon
    return None

def infer_fresh_binary(path: Path):
    """
    Kaggle Food Freshness:
      Dataset/Fresh/FreshApple/...
      Dataset/Rotten/RottenApple/...
    → if a folder/file name starts with 'fresh', it is fresh;
      if it starts with 'rotten', it is rotten.
    """
    name_lower   = path.name.lower()
    parent_lower = path.parent.name.lower()
    grand_lower  = path.parent.parent.name.lower() if path.parent.parent != path.parent else ""

    cands = [name_lower, parent_lower, grand_lower]

    if any(part.startswith("fresh") for part in cands):
        return "fresh"
    if any(part.startswith("rotten") for part in cands):
        return "rotten"
    return None   # discard if ambiguous

# ============================================
# STEP 3. Preprocessing: local → /content/data/freshguard/images + labels.csv
# ============================================
def build_metadata(src_root: Path) -> pd.DataFrame:
    if IMG_ROOT.exists():
        shutil.rmtree(IMG_ROOT)
    IMG_ROOT.mkdir(parents=True, exist_ok=True)

    rows = []
    total = 0
    used  = 0

    print(f"[Prep] {src_root} - scanning images...")

    for p in src_root.rglob("*"):
        if not p.is_file():
            continue
        total += 1
        if p.suffix.lower() not in IMG_EXTS:
            continue

        fruit = infer_fruit(p)
        fresh_label = infer_fresh_binary(p)
        if not fruit or not fresh_label:
            continue

        used += 1
        new_name = f"{uuid.uuid4().hex}{p.suffix.lower()}"
        dst_dir  = IMG_ROOT / fruit / fresh_label
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst_path = dst_dir / new_name
        shutil.copy2(p, dst_path)

        rows.append({
            "path":        str(dst_path),
            "fruit_name":  fruit,
            "fresh_label": fresh_label,  # 'fresh' or 'rotten'
        })

    df = pd.DataFrame(rows)
    META_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(META_CSV, index=False)

    print(f"[Prep] Total files: {total}, images used: {used}")
    if len(df):
        print("\n[Prep] Fruit × Fresh/Rotten distribution:")
        print(df.groupby(["fruit_name", "fresh_label"]).size().unstack(fill_value=0))

    return df

# ============================================
# STEP 4. Dataset & EfficientNet-B0 model definition
# ============================================
class FreshGuardDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return (
            img,
            torch.tensor(int(row["fruit_idx"]), dtype=torch.long),
            torch.tensor(int(row["fresh_idx"]), dtype=torch.long),
        )

class MultiTaskEffB0(nn.Module):
    def __init__(self, n_fruit: int, n_fresh: int):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        backbone = efficientnet_b0(weights=weights)
        in_ch = backbone.classifier[1].in_features
        backbone.classifier[1] = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.2)
        self.head_fruit = nn.Linear(in_ch, n_fruit)
        self.head_fresh = nn.Linear(in_ch, n_fresh)  # 2 classes (fresh / rotten)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        return self.head_fruit(feat), self.head_fresh(feat)

# ============================================
# STEP 5. Training utilities
# ============================================
def compute_class_weight(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    weights = 1.0 / np.log1p(counts + 1.0)
    return torch.tensor(weights / weights.mean(), dtype=torch.float32)

def train_val_test_split(df: pd.DataFrame):
    tag = df["fruit_idx"].astype(str) + "_" + df["fresh_idx"].astype(str)
    train_df, valtest_df = train_test_split(
        df, test_size=0.2, random_state=SEED, stratify=tag
    )
    tag2 = valtest_df["fruit_idx"].astype(str) + "_" + valtest_df["fresh_idx"].astype(str)
    val_df, test_df = train_test_split(
        valtest_df, test_size=0.5, random_state=SEED, stratify=tag2
    )
    return train_df, val_df, test_df

def build_transforms():
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    T_train = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    T_val = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return T_train, T_val

def eval_f1(model, dl):
    model.eval()
    preds_fruit, gts_fruit = [], []
    preds_fresh, gts_fresh = [], []
    with torch.no_grad():
        for x, yf, yb in dl:
            x = x.to(DEVICE)
            yf = yf.to(DEVICE)
            yb = yb.to(DEVICE)

            pf, pb = model(x)
            pf_idx = pf.argmax(1)
            pb_idx = pb.argmax(1)

            preds_fruit += pf_idx.cpu().tolist()
            gts_fruit   += yf.cpu().tolist()
            preds_fresh += pb_idx.cpu().tolist()
            gts_fresh   += yb.cpu().tolist()

    f1_fruit = f1_score(gts_fruit, preds_fruit, average="macro")
    f1_fresh = f1_score(gts_fresh, preds_fresh, average="macro")
    return f1_fruit, f1_fresh

def train_effnet_multitask(meta_csv: Path, out_path: Path, epochs=15, batch_size=64, lr=1e-4):
    df = pd.read_csv(meta_csv)
    df = df[df["fruit_name"].isin(TARGET_FRUITS.keys())]
    df = df[df["fresh_label"].isin(["fresh", "rotten"])]
    df = df.reset_index(drop=True)

    if df.empty:
        raise RuntimeError("No training data found. Re-check the paths/label parsing first.")

    fruit_names  = sorted(df["fruit_name"].unique().tolist())
    fresh_labels = ["fresh", "rotten"]  # binary labels

    fruit2idx = {f: i for i, f in enumerate(fruit_names)}
    fresh2idx = {f: i for i, f in enumerate(fresh_labels)}

    df["fruit_idx"] = df["fruit_name"].map(fruit2idx)
    df["fresh_idx"] = df["fresh_label"].map(fresh2idx)

    print("[Train] fruit2idx:", fruit2idx)
    print("[Train] fresh2idx:", fresh2idx)

    train_df, val_df, test_df = train_val_test_split(df)

    w_fruit = compute_class_weight(train_df["fruit_idx"].values, len(fruit_names)).to(DEVICE)
    w_fresh = compute_class_weight(train_df["fresh_idx"].values, len(fresh_labels)).to(DEVICE)

    T_train, T_val = build_transforms()

    ds_train = FreshGuardDataset(train_df, T_train)
    ds_val   = FreshGuardDataset(val_df,   T_val)
    ds_test  = FreshGuardDataset(test_df,  T_val)

    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=2)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=2)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=2)

    model = MultiTaskEffB0(n_fruit=len(fruit_names), n_fresh=len(fresh_labels)).to(DEVICE)

    crit_fruit = nn.CrossEntropyLoss(weight=w_fruit)
    crit_fresh = nn.CrossEntropyLoss(weight=w_fresh)
    optim = torch.optim.AdamW(model.parameters(), lr=lr)

    best_score = -1

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n_batch = 0

        for i, (x, yf, yb) in enumerate(dl_train):
            x, yf, yb = x.to(DEVICE), yf.to(DEVICE), yb.to(DEVICE)
            optim.zero_grad()
            pf, pb = model(x)
            loss = crit_fruit(pf, yf) + crit_fresh(pb, yb)
            loss.backward()
            optim.step()

            running_loss += loss.item()
            n_batch += 1

            if (i + 1) % PRINT_EVERY == 0:
                print(f"[Epoch {epoch:02d}] Step {i+1}/{len(dl_train)} loss={running_loss/n_batch:.4f}")

        avg_loss = running_loss / max(1, n_batch)
        val_f1_fruit, val_f1_fresh = eval_f1(model, dl_val)
        score = val_f1_fruit + val_f1_fresh
        print(f"[Epoch {epoch:02d}] train_loss={avg_loss:.4f} | "
              f"val_f1_fruit={val_f1_fruit:.4f}, val_f1_fresh={val_f1_fresh:.4f}, sum={score:.4f}")

        if score > best_score:
            best_score = score
            torch.save(
                {
                    "model_state":  model.state_dict(),
                    "fruit_names":  fruit_names,
                    "fresh_labels": fresh_labels,  # ['fresh', 'rotten']
                },
                out_path,
            )
            print(f"  >>> Saved new best model: {out_path}")

    print(f"[Train] Final best score (val_fruit + val_fresh) = {best_score:.4f}")

    # Evaluate on the test set
    ckpt = torch.load(out_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    test_f1_fruit, test_f1_fresh = eval_f1(model, dl_test)
    print(f"[Test] F1(fruit)={test_f1_fruit:.4f}, F1(fresh)={test_f1_fresh:.4f}")

def main_train():
    src_root = resolve_source_dir()
    build_metadata(src_root)
    train_effnet_multitask(META_CSV, EFFNET_BEST_PATH, epochs=15, batch_size=64, lr=1e-4)

print("=== EfficientNet-B0 multitask training started ===")
main_train()
print("=== EfficientNet-B0 training complete ===")

# ============================================
# STEP 6. YOLO + EfficientNet pipeline (3-level Fresh/Normal/Rotten output)
# ============================================
import cv2
from ultralytics import YOLO

# YOLO weight path 
YOLO_WEIGHTS = Path("/content/drive/MyDrive/FreshGuard/yolo/best.pt")

# EfficientNet training result (the file trained above)
EFFNET_BEST_PATH = EFFNET_BEST_PATH  # reuse as-is

# YOLO class id → fruit name mapping
YOLO_ID_TO_FRUIT = {
    0: "apple",
    1: "banana",
    2: "orange",
    3: "strawberry",
    4: "tomato",
    5: "cucumber",
    6: "bell_pepper",
    7: "mango",
    8: "potato",
    9: "carrot",
}

# Load EfficientNet
ckpt = torch.load(EFFNET_BEST_PATH, map_location=DEVICE)
fruit_names  = ckpt["fruit_names"]
fresh_labels = ckpt["fresh_labels"]  # ['fresh', 'rotten']
fresh_label2idx = {lab: i for i, lab in enumerate(fresh_labels)}

effnet_model = MultiTaskEffB0(n_fruit=len(fruit_names), n_fresh=len(fresh_labels)).to(DEVICE)
effnet_model.load_state_dict(ckpt["model_state"])
effnet_model.eval()

# Transform for EfficientNet inference (same as val)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
effnet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Map fresh probability → 3 levels: fresh/normal/rotten
THRESH_FRESH_HIGH = 0.7
THRESH_FRESH_LOW  = 0.3

def prob_to_3class(p_fresh: float) -> str:
    if p_fresh >= THRESH_FRESH_HIGH:
        return "fresh"
    elif p_fresh <= THRESH_FRESH_LOW:
        return "rotten"
    else:
        return "normal"

def classify_crop_bgr(img_bgr):
    """YOLO crop (BGR) → EfficientNet: fruit + fresh/normal/rotten"""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    x = effnet_transform(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pf, pb = effnet_model(x)
        fruit_idx = pf.argmax(1).item()

        probs = torch.softmax(pb, dim=1)[0].cpu().numpy()
        p_fresh = float(probs[fresh_label2idx["fresh"]])
        fresh_3 = prob_to_3class(p_fresh)

    return fruit_names[fruit_idx], fresh_3, p_fresh

# Load YOLO
yolo_model = YOLO(str(YOLO_WEIGHTS))

def demo_on_camera():
    """Real-time webcam demo of the YOLO + EfficientNet pipeline"""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Cannot open the camera.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo_model(frame, verbose=False)

        for r in results:
            boxes = r.boxes
            for box in boxes:
                cls_id = int(box.cls.item())
                conf = float(box.conf.item())

                if cls_id not in YOLO_ID_TO_FRUIT:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                h, w, _ = frame.shape
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                fruit_pred, fresh_3, p_fresh = classify_crop_bgr(crop)
                label = f"{fruit_pred} / {fresh_3} (p_fresh={p_fresh:.2f}, {conf:.2f})"

                if fresh_3 == "fresh":
                    color = (0, 255, 0)
                elif fresh_3 == "normal":
                    color = (0, 255, 255)
                else:
                    color = (0, 0, 255)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame, label, (x1, max(0, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA
                )

        cv2.imshow("FreshGuard - YOLO + EfficientNet", frame)
        key = cv2.waitKey(1)
        if key == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

def demo_on_image(image_path: str):
    """Test the YOLO + EfficientNet pipeline on a single image file"""
    frame = cv2.imread(image_path)
    if frame is None:
        print("Cannot read the image:", image_path)
        return

    results = yolo_model(frame, verbose=False)
    for r in results:
        boxes = r.boxes
        for box in boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            if cls_id not in YOLO_ID_TO_FRUIT:
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            h, w, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            fruit_pred, fresh_3, p_fresh = classify_crop_bgr(crop)
            label = f"{fruit_pred} / {fresh_3} (p_fresh={p_fresh:.2f}, {conf:.2f})"

            if fresh_3 == "fresh":
                color = (0, 255, 0)
            elif fresh_3 == "normal":
                color = (0, 255, 255)
            else:
                color = (0, 0, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame, label, (x1, max(0, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA
            )

    cv2.imshow("FreshGuard - YOLO + EfficientNet (Image)", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

print("=== YOLO + EfficientNet pipeline loaded ===")
print("Webcam demo: demo_on_camera()")
print("Image demo: demo_on_image('/content/test.jpg')")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== EfficientNet-B0 multitask training started ===
[Data] Using Drive zip: /content/drive/MyDrive/datasets/food-freshness-dataset.zip
[Data] Zip extraction complete: /content/data/raw/food_freshness
[Prep] /content/data/raw/food_freshness - scanning images...
[Prep] Total files: 71322, images used: 69105

[Prep] Fruit × Fresh/Rotten distribution:
fresh_label  fresh  rotten
fruit_name                
apple         3431    4437
banana        3473    4038
bell_pepper   1045    1492
carrot        9899    2419
cucumber      1104    1014
mango          605     593
orange       10831    3292
potato        1151    1387
strawberry     603     596
tomato       13681    4014
[Train] fruit2idx: {'apple': 0, 'banana': 1, 'bell_pepper': 2, 'carrot': 3, 'cucumber': 4, 'mango': 5, 'orange': 6, 'potato': 7, 'strawberry': 8, 'tomato': 9}
[Train] fresh2idx: {'fresh': 0, 'rotten

100%|██████████| 20.5M/20.5M [00:00<00:00, 105MB/s]


[Epoch 01] Step 50/864 loss=2.1464
[Epoch 01] Step 100/864 loss=1.5969
[Epoch 01] Step 150/864 loss=1.2783


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[Epoch 01] Step 200/864 loss=1.0712
[Epoch 01] Step 250/864 loss=0.9279
[Epoch 01] Step 300/864 loss=0.8228
[Epoch 01] Step 350/864 loss=0.7452
[Epoch 01] Step 400/864 loss=0.6817


...

[Epoch 07] Step 150/864 loss=0.0371
[Epoch 07] Step 200/864 loss=0.0359
[Epoch 07] Step 250/864 loss=0.0360
[Epoch 07] Step 300/864 loss=0.0365
[Epoch 07] Step 350/864 loss=0.0363
[Epoch 07] Step 400/864 loss=0.0363
[Epoch 07] Step 450/864 loss=0.0369
[Epoch 07] Step 500/864 loss=0.0364
[Epoch 07] Step 550/864 loss=0.0367
[Epoch 07] Step 600/864 loss=0.0367
[Epoch 07] Step 650/864 loss=0.0373
[Epoch 07] Step 700/864 loss=0.0381
[Epoch 07] Step 750/864 loss=0.0383
[Epoch 07] Step 800/864 loss=0.0379
[Epoch 07] Step 850/864 loss=0.0384
[Epoch 07] train_loss=0.0386 | val_f1_fruit=0.9904, val_f1_fresh=0.9792, sum=1.9696


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[Epoch 08] Step 50/864 loss=0.0341
[Epoch 08] Step 100/864 loss=0.0349
[Epoch 08] Step 150/864 loss=0.0345
[Epoch 08] Step 200/864 loss=0.0347
[Epoch 08] Step 250/864 loss=0.0337
[Epoch 08] Step 300/864 loss=0.0339
[Epoch 08] Step 350/864 loss=0.0357


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[Epoch 08] Step 400/864 loss=0.0357
[Epoch 08] Step 450/864 loss=0.0356
[Epoch 08] Step 500/864 loss=0.0358
[Epoch 08] Step 550/864 loss=0.0356
[Epoch 08] Step 600/864 loss=0.0359
[Epoch 08] Step 650/864 loss=0.0358
[Epoch 08] Step 700/864 loss=0.0354
[Epoch 08] Step 750/864 loss=0.0357
[Epoch 08] Step 800/864 loss=0.0356


# =========================================
# STEP 3. YOLO + EfficientNet pipeline
#  - YOLO: multi-fruit detection (best.pt stored in Drive)
#  - EfficientNet: cropped box → fruit + freshness prediction
# =========================================

In [ ]:
# =========================================
#  YOLO + EfficientNet pipeline
#  - YOLO: multi-fruit detection (best.pt stored in Drive)
#  - EfficientNet: cropped box → fruit + Fresh/Rotten probability
#  - The UI shows 3 levels: fresh/normal/rotten
# =========================================
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from ultralytics import YOLO
from torchvision import transforms
from PIL import Image
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# ===== YOLO weight path =====
YOLO_WEIGHTS = Path("/content/drive/MyDrive/FreshGuard/yolo/best.pt")

# EfficientNet training result (saved in STEP 2)
EFFNET_BEST_PATH = Path("/content/drive/MyDrive/FreshGuard/effnet/efficientnet_b0_freshguard_multitask.pt")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# YOLO class id → fruit name mapping
# (adjust to match your YOLO training class order)
YOLO_ID_TO_FRUIT = {
    0: "apple",
    1: "banana",
    2: "orange",
    3: "strawberry",
    4: "tomato",
    5: "cucumber",
    6: "bell_pepper",
    7: "mango",
    8: "potato",
    9: "carrot",
}

# EfficientNet-B0 architecture (same as in training)
class MultiTaskEffB0(nn.Module):
    def __init__(self, n_fruit: int, n_fresh: int):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        backbone = efficientnet_b0(weights=weights)
        in_ch = backbone.classifier[1].in_features
        backbone.classifier[1] = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.2)
        self.head_fruit = nn.Linear(in_ch, n_fruit)
        self.head_fresh = nn.Linear(in_ch, n_fresh)  # 2 classes

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        return self.head_fruit(feat), self.head_fresh(feat)

# Load EfficientNet
ckpt = torch.load(EFFNET_BEST_PATH, map_location=DEVICE)
fruit_names = ckpt["fruit_names"]     # list of fruit classes
fresh_labels = ckpt["fresh_labels"]   # ['fresh', 'rotten']

effnet_model = MultiTaskEffB0(n_fruit=len(fruit_names), n_fresh=len(fresh_labels)).to(DEVICE)
effnet_model.load_state_dict(ckpt["model_state"])
effnet_model.eval()

# Transform for EfficientNet inference (same normalization as training)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
effnet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Load YOLO
yolo_model = YOLO(str(YOLO_WEIGHTS))

# Thresholds that split Fresh/Normal/Rotten ranges
FRESH_HIGH = 0.7   # p(fresh) >= 0.7 → fresh
FRESH_LOW  = 0.3   # p(fresh) <= 0.3 → rotten
# in between → normal

def map_prob_to_3level(p_fresh: float) -> str:
    if p_fresh >= FRESH_HIGH:
        return "fresh"
    if p_fresh <= FRESH_LOW:
        return "rotten"
    return "normal"

def classify_crop_bgr(img_bgr):
    """BGR crop → EfficientNet predicts fruit + fresh/normal/rotten"""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    x = effnet_transform(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pf, pb = effnet_model(x)
        # fruit
        fruit_idx = pf.argmax(1).item()
        # Fresh/Rotten probability
        probs = F.softmax(pb, dim=1)[0].cpu().numpy()
        idx_fresh = fresh_labels.index("fresh")  # should be 0
        p_fresh = float(probs[idx_fresh])
        level3 = map_prob_to_3level(p_fresh)

    return fruit_names[fruit_idx], level3, p_fresh

def demo_on_camera():
    """Real-time webcam demo of the YOLO + EfficientNet pipeline"""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Cannot open the camera.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # YOLO inference
        results = yolo_model(frame, verbose=False)

        for r in results:
            boxes = r.boxes
            for box in boxes:
                cls_id = int(box.cls.item())
                conf = float(box.conf.item())

                if cls_id not in YOLO_ID_TO_FRUIT:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                h, w, _ = frame.shape
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                fruit_pred, fresh_level3, p_fresh = classify_crop_bgr(crop)
                label = f"{fruit_pred} / {fresh_level3} (p_fresh={p_fresh:.2f}, conf={conf:.2f})"

                if fresh_level3 == "fresh":
                    color = (0, 255, 0)
                elif fresh_level3 == "normal":
                    color = (0, 255, 255)
                else:  # rotten
                    color = (0, 0, 255)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame, label, (x1, max(0, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA
                )

        cv2.imshow("FreshGuard - YOLO + EfficientNet (Camera)", frame)
        key = cv2.waitKey(1)
        if key == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

def demo_on_image(image_path: str):
    """Test the YOLO + EfficientNet pipeline on a single image file"""
    frame = cv2.imread(image_path)
    if frame is None:
        print("Cannot read the image:", image_path)
        return

    results = yolo_model(frame, verbose=False)
    for r in results:
        boxes = r.boxes
        for box in boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            if cls_id not in YOLO_ID_TO_FRUIT:
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            h, w, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            fruit_pred, fresh_level3, p_fresh = classify_crop_bgr(crop)
            label = f"{fruit_pred} / {fresh_level3} (p_fresh={p_fresh:.2f}, conf={conf:.2f})"

            if fresh_level3 == "fresh":
                color = (0, 255, 0)
            elif fresh_level3 == "normal":
                color = (0, 255, 255)
            else:
                color = (0, 0, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame, label, (x1, max(0, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA
            )

    cv2.imshow("FreshGuard - YOLO + EfficientNet (Image)", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

print("=== YOLO + EfficientNet pipeline loaded ===")
print("Run webcam demo: demo_on_camera()")
print("Run image demo: demo_on_image('/content/test.jpg')")
